# Vector Store Base Interfaces

The `base.py` module defines the common interface for vector stores and the retriever wrapper used to query them.

A vector store stores embedded documents and supports similarity-based retrieval. The base interface provides synchronous and asynchronous document ingestion, deletion, ID lookup, similarity search, relevance-score search, maximal marginal relevance search, construction helpers, and retriever conversion.

## Type Variables

1. `VST`: Represents a concrete subclass of `VectorStore`.
   * **Definition:**
     ```python
     VST = TypeVar(
         "VST",
         bound="VectorStore"
     )
     ```

# VectorStore

`VectorStore` is the abstract base interface implemented by LangChain vector-store integrations.

Subclasses must implement `similarity_search` and `from_texts`. Depending on the features supported by the underlying store, they may also implement native document ingestion, deletion, ID lookup, score-based search, vector-based search, maximal marginal relevance search, and asynchronous variants.

## Bases

- `ABC`

### Properties

1. `embeddings`: Returns the embedding object used for query generation when the vector store exposes one.

   The base implementation logs a debug message and returns `None`.

   * **Type:**
     ```python
     embeddings: Embeddings | None
     ```

### Methods

1. `add_texts`: Embeds and adds text values to the vector store.

   When a subclass implements `add_documents` but does not override `add_texts`, the base implementation converts each text, metadata dictionary, and optional ID into a `Document` and delegates to `add_documents`.

   A `ValueError` is raised when the number of metadata dictionaries does not match the number of texts. A `NotImplementedError` is raised when neither text nor document ingestion is implemented.

   * **Syntax:**
     ```python
     add_texts(
         self,
         texts: Iterable[str], # Text values to add
         metadatas: list[
             dict[str, Any]
         ] | None = None, # Optional metadata for each text
         *,
         ids: list[str] | None = None, # Optional IDs for the text values
         **kwargs: Any # Vector-store-specific parameters
     ) -> list[str]
     ```

2. `delete`: Deletes entries by vector ID or other vector-store-specific criteria.

   Subclasses must override this method to support deletion. The base implementation raises `NotImplementedError`.

   * **Syntax:**
     ```python
     delete(
         self,
         ids: list[str] | None = None, # IDs to delete, or None to delete all
         **kwargs: Any # Additional deletion criteria
     ) -> bool | None
     ```

3. `get_by_ids`: Returns documents matching the supplied IDs.

   Fewer documents may be returned when IDs are missing or duplicated. The returned order is not required to match the input order, so callers should inspect each `Document.id`.

   Implementations should not raise an exception merely because some requested IDs are absent. The base implementation raises `NotImplementedError`.

   * **Syntax:**
     ```python
     get_by_ids(
         self,
         ids: Sequence[str], / # Document IDs to retrieve
     ) -> list[Document]
     ```

4. `aget_by_ids`: Asynchronously returns documents matching the supplied IDs.

   The base implementation runs `get_by_ids` in an executor. Subclasses may override it with a native asynchronous implementation.

   * **Syntax:**
     ```python
     async aget_by_ids(
         self,
         ids: Sequence[str], / # Document IDs to retrieve
     ) -> list[Document]
     ```

5. `adelete`: Asynchronously deletes entries by ID or other criteria.

   The base implementation runs `delete` in an executor.

   * **Syntax:**
     ```python
     async adelete(
         self,
         ids: list[str] | None = None, # IDs to delete, or None to delete all
         **kwargs: Any # Additional deletion criteria
     ) -> bool | None
     ```

6. `aadd_texts`: Asynchronously embeds and adds text values to the vector store.

   When a subclass implements `aadd_documents`, the texts are converted to `Document` objects and passed to that method. Otherwise, the synchronous `add_texts` method is executed in an executor.

   * **Syntax:**
     ```python
     async aadd_texts(
         self,
         texts: Iterable[str], # Text values to add
         metadatas: list[
             dict[str, Any]
         ] | None = None, # Optional metadata for each text
         *,
         ids: list[str] | None = None, # Optional IDs for the text values
         **kwargs: Any # Vector-store-specific parameters
     ) -> list[str]
     ```

7. `add_documents`: Adds or updates `Document` objects in the vector store.

   When a subclass implements `add_texts` but not `add_documents`, the base implementation extracts page content, metadata, and IDs from the documents and delegates to `add_texts`. Explicit IDs supplied through `kwargs` take precedence over document IDs.

   A `NotImplementedError` is raised when neither document nor text ingestion is implemented.

   * **Syntax:**
     ```python
     add_documents(
         self,
         documents: list[Document], # Documents to add or update
         **kwargs: Any # Additional vector-store parameters
     ) -> list[str]
     ```

8. `aadd_documents`: Asynchronously adds or updates documents.

   When a subclass implements `aadd_texts`, the base implementation extracts document content, metadata, and IDs and delegates to it. Otherwise, `add_documents` is executed in an executor.

   * **Syntax:**
     ```python
     async aadd_documents(
         self,
         documents: list[Document], # Documents to add or update
         **kwargs: Any # Additional vector-store parameters
     ) -> list[str]
     ```

9. `search`: Performs one of the supported synchronous search modes.

   The supported values are `"similarity"`, `"similarity_score_threshold"`, and `"mmr"`. Threshold search discards the relevance-score values after filtering.

   A `ValueError` is raised for an unsupported search type.

   * **Syntax:**
     ```python
     search(
         self,
         query: str, # Query text
         search_type: str, # Search mode to perform
         **kwargs: Any # Arguments passed to the selected search method
     ) -> list[Document]
     ```

10. `asearch`: Performs one of the supported asynchronous search modes.

    It dispatches to `asimilarity_search`, `asimilarity_search_with_relevance_scores`, or `amax_marginal_relevance_search`.

    A `ValueError` is raised for an unsupported search type.

    * **Syntax:**
      ```python
      async asearch(
          self,
          query: str, # Query text
          search_type: str, # Search mode to perform
          **kwargs: Any # Arguments passed to the selected search method
      ) -> list[Document]
      ```

11. `similarity_search`: Returns documents most similar to a query.

    Subclasses must implement this abstract method.

    * **Syntax:**
      ```python
      @abstractmethod
      similarity_search(
          self,
          query: str, # Query text
          k: int = 4, # Maximum number of documents to return
          **kwargs: Any # Vector-store-specific search parameters
      ) -> list[Document]
      ```

12. `_euclidean_relevance_score_fn`: Converts Euclidean distance between normalized embeddings into a relevance score.

    A distance of `0` maps to the highest relevance. The function assumes the maximum relevant normalized Euclidean distance is `sqrt(2)`.

    * **Syntax:**
      ```python
      @staticmethod
      _euclidean_relevance_score_fn(
          distance: float # Euclidean distance
      ) -> float
      ```

13. `_cosine_relevance_score_fn`: Converts cosine distance into a relevance score by subtracting it from `1`.

    * **Syntax:**
      ```python
      @staticmethod
      _cosine_relevance_score_fn(
          distance: float # Cosine distance
      ) -> float
      ```

14. `_max_inner_product_relevance_score_fn`: Converts a maximum-inner-product distance into a relevance score.

    Positive values are subtracted from `1`, while non-positive values are negated.

    * **Syntax:**
      ```python
      @staticmethod
      _max_inner_product_relevance_score_fn(
          distance: float # Inner-product distance value
      ) -> float
      ```

15. `_select_relevance_score_fn`: Selects the distance-to-relevance conversion function used by the vector store.

    Subclasses that support relevance-score search should override this protected method according to their distance metric and embedding scale. The base implementation raises `NotImplementedError`.

    * **Syntax:**
      ```python
      _select_relevance_score_fn(
          self
      ) -> Callable[
          [float],
          float
      ]
      ```

16. `similarity_search_with_score`: Returns documents together with raw vector-store similarity or distance scores.

    Subclasses must override this method to support score-based search. The base implementation raises `NotImplementedError`.

    * **Syntax:**
      ```python
      similarity_search_with_score(
          self,
          *args: Any, # Positional search arguments
          **kwargs: Any # Keyword search arguments
      ) -> list[
          tuple[
              Document,
              float
          ]
      ]
      ```

17. `asimilarity_search_with_score`: Asynchronously returns documents with raw scores.

    The base implementation executes `similarity_search_with_score` in an executor.

    * **Syntax:**
      ```python
      async asimilarity_search_with_score(
          self,
          *args: Any, # Positional search arguments
          **kwargs: Any # Keyword search arguments
      ) -> list[
          tuple[
              Document,
              float
          ]
      ]
      ```

18. `_similarity_search_with_relevance_scores`: Converts raw synchronous scores into relevance scores between `0` and `1`.

    It uses the function returned by `_select_relevance_score_fn`. Subclasses may override this protected method when their scoring system requires different handling.

    * **Syntax:**
      ```python
      _similarity_search_with_relevance_scores(
          self,
          query: str, # Query text
          k: int = 4, # Maximum number of documents to return
          **kwargs: Any # Search parameters
      ) -> list[
          tuple[
              Document,
              float
          ]
      ]
      ```

19. `_asimilarity_search_with_relevance_scores`: Converts raw asynchronous scores into relevance scores between `0` and `1`.

    It uses `_select_relevance_score_fn` and `asimilarity_search_with_score`.

    * **Syntax:**
      ```python
      async _asimilarity_search_with_relevance_scores(
          self,
          query: str, # Query text
          k: int = 4, # Maximum number of documents to return
          **kwargs: Any # Search parameters
      ) -> list[
          tuple[
              Document,
              float
          ]
      ]
      ```

20. `similarity_search_with_relevance_scores`: Returns documents and normalized relevance scores.

    A `score_threshold` value supplied through `kwargs` filters out results below that threshold. A warning is emitted when a returned score falls outside the expected range of `0` to `1`. A log warning is emitted when threshold filtering produces no results.

    * **Syntax:**
      ```python
      similarity_search_with_relevance_scores(
          self,
          query: str, # Query text
          k: int = 4, # Maximum number of documents to return
          **kwargs: Any # Search parameters, including optional score_threshold
      ) -> list[
          tuple[
              Document,
              float
          ]
      ]
      ```

21. `asimilarity_search_with_relevance_scores`: Asynchronously returns documents and normalized relevance scores.

    It applies the same range warning and optional threshold filtering as the synchronous method.

    * **Syntax:**
      ```python
      async asimilarity_search_with_relevance_scores(
          self,
          query: str, # Query text
          k: int = 4, # Maximum number of documents to return
          **kwargs: Any # Search parameters, including optional score_threshold
      ) -> list[
          tuple[
              Document,
              float
          ]
      ]
      ```

22. `asimilarity_search`: Asynchronously returns documents most similar to a query.

    The base implementation runs `similarity_search` in an executor.

    * **Syntax:**
      ```python
      async asimilarity_search(
          self,
          query: str, # Query text
          k: int = 4, # Maximum number of documents to return
          **kwargs: Any # Vector-store-specific search parameters
      ) -> list[Document]
      ```

23. `similarity_search_by_vector`: Returns documents most similar to an embedding vector.

    Subclasses must override this method to support vector-based search. The base implementation raises `NotImplementedError`.

    * **Syntax:**
      ```python
      similarity_search_by_vector(
          self,
          embedding: list[float], # Query embedding vector
          k: int = 4, # Maximum number of documents to return
          **kwargs: Any # Vector-store-specific search parameters
      ) -> list[Document]
      ```

24. `asimilarity_search_by_vector`: Asynchronously returns documents most similar to an embedding vector.

    The base implementation executes `similarity_search_by_vector` in an executor.

    * **Syntax:**
      ```python
      async asimilarity_search_by_vector(
          self,
          embedding: list[float], # Query embedding vector
          k: int = 4, # Maximum number of documents to return
          **kwargs: Any # Vector-store-specific search parameters
      ) -> list[Document]
      ```

25. `max_marginal_relevance_search`: Returns documents selected through maximal marginal relevance.

    MMR balances similarity to the query against diversity among the selected documents. `lambda_mult=1` prioritizes similarity, while `lambda_mult=0` prioritizes diversity.

    Subclasses must override this method to support MMR. The base implementation raises `NotImplementedError`.

    * **Syntax:**
      ```python
      max_marginal_relevance_search(
          self,
          query: str, # Query text
          k: int = 4, # Number of documents to return
          fetch_k: int = 20, # Candidate documents considered by MMR
          lambda_mult: float = 0.5, # Similarity-versus-diversity weighting
          **kwargs: Any # Vector-store-specific search parameters
      ) -> list[Document]
      ```

26. `amax_marginal_relevance_search`: Asynchronously performs maximal marginal relevance search.

    The base implementation executes `max_marginal_relevance_search` in an executor.

    * **Syntax:**
      ```python
      async amax_marginal_relevance_search(
          self,
          query: str, # Query text
          k: int = 4, # Number of documents to return
          fetch_k: int = 20, # Candidate documents considered by MMR
          lambda_mult: float = 0.5, # Similarity-versus-diversity weighting
          **kwargs: Any # Vector-store-specific search parameters
      ) -> list[Document]
      ```

27. `max_marginal_relevance_search_by_vector`: Performs maximal marginal relevance search using an embedding vector.

    Subclasses must override this method to support vector-based MMR. The base implementation raises `NotImplementedError`.

    * **Syntax:**
      ```python
      max_marginal_relevance_search_by_vector(
          self,
          embedding: list[float], # Query embedding vector
          k: int = 4, # Number of documents to return
          fetch_k: int = 20, # Candidate documents considered by MMR
          lambda_mult: float = 0.5, # Similarity-versus-diversity weighting
          **kwargs: Any # Vector-store-specific search parameters
      ) -> list[Document]
      ```

28. `amax_marginal_relevance_search_by_vector`: Asynchronously performs vector-based maximal marginal relevance search.

    The base implementation executes `max_marginal_relevance_search_by_vector` in an executor.

    * **Syntax:**
      ```python
      async amax_marginal_relevance_search_by_vector(
          self,
          embedding: list[float], # Query embedding vector
          k: int = 4, # Number of documents to return
          fetch_k: int = 20, # Candidate documents considered by MMR
          lambda_mult: float = 0.5, # Similarity-versus-diversity weighting
          **kwargs: Any # Vector-store-specific search parameters
      ) -> list[Document]
      ```

29. `from_documents`: Constructs a vector store from `Document` objects.

    The method extracts page content and metadata and delegates to `from_texts`. Document IDs are forwarded when at least one document contains an ID and explicit IDs were not supplied in `kwargs`.

    * **Syntax:**
      ```python
      @classmethod
      from_documents(
          cls,
          documents: list[Document], # Documents used to initialize the store
          embedding: Embeddings, # Embedding implementation
          **kwargs: Any # Additional construction parameters
      ) -> Self
      ```

30. `afrom_documents`: Asynchronously constructs a vector store from documents.

    It extracts content, metadata, and optional IDs and delegates to `afrom_texts`.

    * **Syntax:**
      ```python
      @classmethod
      async afrom_documents(
          cls,
          documents: list[Document], # Documents used to initialize the store
          embedding: Embeddings, # Embedding implementation
          **kwargs: Any # Additional construction parameters
      ) -> Self
      ```

31. `from_texts`: Constructs a vector store from text values and embeddings.

    Concrete vector-store classes must implement this abstract class method.

    * **Syntax:**
      ```python
      @classmethod
      @abstractmethod
      from_texts(
          cls: type[VST],
          texts: list[str], # Text values used to initialize the store
          embedding: Embeddings, # Embedding implementation
          metadatas: list[
              dict[str, Any]
          ] | None = None, # Optional metadata for each text
          *,
          ids: list[str] | None = None, # Optional IDs for the text values
          **kwargs: Any # Additional construction parameters
      ) -> VST
      ```

32. `afrom_texts`: Asynchronously constructs a vector store from text values and embeddings.

    The base implementation executes `from_texts` in an executor.

    * **Syntax:**
      ```python
      @classmethod
      async afrom_texts(
          cls,
          texts: list[str], # Text values used to initialize the store
          embedding: Embeddings, # Embedding implementation
          metadatas: list[
              dict[str, Any]
          ] | None = None, # Optional metadata for each text
          *,
          ids: list[str] | None = None, # Optional IDs for the text values
          **kwargs: Any # Additional construction parameters
      ) -> Self
      ```

33. `_get_retriever_tags`: Returns default tags for a retriever created from this vector store.

    The vector-store class name is always included. The embeddings class name is included when the `embeddings` property is available.

    * **Syntax:**
      ```python
      _get_retriever_tags(
          self
      ) -> list[str]
      ```

34. `as_retriever`: Creates a `VectorStoreRetriever` backed by this vector store.

    `search_type` may be `"similarity"`, `"similarity_score_threshold"`, or `"mmr"`. `search_kwargs` may include values such as `k`, `score_threshold`, `fetch_k`, `lambda_mult`, and metadata filters.

    When explicit tags are not supplied, default vector-store and embedding tags are generated through `_get_retriever_tags`.

    * **Syntax:**
      ```python
      as_retriever(
          self,
          **kwargs: Any # Retriever configuration and search parameters
      ) -> VectorStoreRetriever
      ```

# VectorStoreRetriever

`VectorStoreRetriever` adapts a `VectorStore` to the LangChain retriever interface.

It validates the selected search mode and delegates retrieval to the corresponding synchronous or asynchronous vector-store method.

## Bases

- `BaseRetriever`

## Attributes

1. `vectorstore`: Stores the vector store used for retrieval.
   * **Type:**
     ```python
     vectorstore: VectorStore
     ```

2. `search_type`: Stores the search mode used by the retriever.
   * **Type:**
     ```python
     search_type: str = "similarity"
     ```

3. `search_kwargs`: Stores arguments passed to the selected vector-store search method.
   * **Type:**
     ```python
     search_kwargs: dict[
         str,
         Any
     ] = Field(
         default_factory=dict
     )
     ```

4. `allowed_search_types`: Stores the supported search modes.
   * **Type:**
     ```python
     allowed_search_types: ClassVar[
         Collection[str]
     ] = (
         "similarity",
         "similarity_score_threshold",
         "mmr",
     )
     ```

## Configuration

1. `model_config`: Allows arbitrary Python types in the Pydantic model.
   * **Definition:**
     ```python
     model_config = ConfigDict(
         arbitrary_types_allowed=True
     )
     ```

### Methods

1. `validate_search_type`: Validates the retriever's search configuration before model construction.

   A `ValueError` is raised when `search_type` is unsupported. For `"similarity_score_threshold"`, `search_kwargs["score_threshold"]` must be present and must be a `float`.

   * **Syntax:**
     ```python
     @model_validator(
         mode="before"
     )
     @classmethod
     validate_search_type(
         cls,
         values: dict[
             str,
             Any
         ] # Values supplied to the model
     ) -> Any
     ```

2. `_get_ls_params`: Returns standard LangSmith parameters for tracing retrieval.

   It merges `search_kwargs` with invocation-specific arguments, adds the vector-store provider class name, and adds an embedding-provider class name when one can be detected.

   * **Syntax:**
     ```python
     _get_ls_params(
         self,
         **kwargs: Any # Invocation-specific retrieval parameters
     ) -> LangSmithRetrieverParams
     ```

3. `_get_relevant_documents`: Retrieves documents synchronously.

   Search arguments supplied at invocation override values with the same keys in `search_kwargs`. The method dispatches to similarity, relevance-threshold, or MMR search based on `search_type`.

   * **Syntax:**
     ```python
     _get_relevant_documents(
         self,
         query: str, # Retrieval query
         *,
         run_manager: CallbackManagerForRetrieverRun, # Retriever callback manager
         **kwargs: Any # Invocation-specific search parameters
     ) -> list[Document]
     ```

4. `_aget_relevant_documents`: Retrieves documents asynchronously.

   It follows the same search-mode dispatch and argument-merging rules as `_get_relevant_documents`.

   * **Syntax:**
     ```python
     async _aget_relevant_documents(
         self,
         query: str, # Retrieval query
         *,
         run_manager: AsyncCallbackManagerForRetrieverRun, # Async retriever callback manager
         **kwargs: Any # Invocation-specific search parameters
     ) -> list[Document]
     ```

5. `add_documents`: Adds documents through the underlying vector store.
   * **Syntax:**
     ```python
     add_documents(
         self,
         documents: list[Document], # Documents to add
         **kwargs: Any # Additional vector-store parameters
     ) -> list[str]
     ```

6. `aadd_documents`: Asynchronously adds documents through the underlying vector store.
   * **Syntax:**
     ```python
     async aadd_documents(
         self,
         documents: list[Document], # Documents to add
         **kwargs: Any # Additional vector-store parameters
     ) -> list[str]
     ```